# Decorators & Logging: 

Concepts: 

● Functions as first-class objects and closures  

● Writing a basic decorator with `@`  

● Decorators that accept arguments  

● `functools.wraps` — preserving function metadata  

● Stacking multiple decorators  

● Practical uses: timing, access control, input validation  

● Why `logging` beats `print` for real projects  

● Log levels: `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`  

● Configuring with `basicConfig`: level, format, file output  

● Named loggers with `logging.getLogger()` 

● File and console handlers running simultaneously 

- Task: 

● API Middleware Simulator: A backend team needs reusable function wrappers that 
handle execution timing, input/output logging, and role-based access control — without 
touching the original functions. All wrappers must be stackable and all activity must go 
through the `logging` module, not print. 

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
    force=True
)

In [2]:
#  decorator for measuring the time execution\
import time 
from functools import wraps
def timing(func):
    @wraps(func)
    def wrapper(*args , **kwargs):
        start = time.time()

        result = func(*args ,**kwargs)

        end = time.time()

        logging.info(f"{func.__name__} executed in {end - start:.4f} seconds")

        return result 
    return wrapper

In [3]:
#  decorator for input/output logging
def log_io(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logging.info(f"{func.__name__} with args={args} and kwargs ={kwargs}")
        result = func(*args , **kwargs)
        logging.info(f"{func.__name__} with result {result}")
        return result
    return wrapper

In [4]:
# decorator for role base 
def require_role(role):

    def decorator(func):

        @wraps(func)
        def wrapper(user_role, *args, **kwargs):

            if user_role != role:
                logging.error("Access denied")
                return None

            return func(user_role, *args, **kwargs)

        return wrapper

    return decorator

In [5]:
#  stackable decorators
@timing
@log_io
@require_role("admin")
def delete_user(user_role, username):

    return f"User {username} deleted"

In [6]:
delete_user("admin" ,"Ali")

INFO: delete_user with args=('admin', 'Ali') and kwargs ={}
INFO: delete_user with result User Ali deleted
INFO: delete_user executed in 0.0044 seconds


'User Ali deleted'

In [7]:
delete_user("guest", "Ali")

INFO: delete_user with args=('guest', 'Ali') and kwargs ={}
ERROR: Access denied
INFO: delete_user with result None
INFO: delete_user executed in 0.0053 seconds
